In [8]:
import os
import numpy as np
import librosa
from gammatone.gtgram import gtgram
from PIL import Image
from tqdm import tqdm

# ---------------- CONFIG ----------------
INPUT_ROOT = "/workspace/Dataset/"
OUTPUT_ROOT = "/workspace/work2-rgb/"
IMG_SIZE = (224, 224)
SR = 16000

N_FFT = 1024
HOP_LENGTH = 512
N_MELS = 128
GT_CHANNELS = 128
# ---------------------------------------

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# ---------- Utility Functions ----------
def normalize(spec):
    spec = spec - spec.min()
    return spec / (spec.max() + 1e-8)

def resize(spec):
    img = Image.fromarray((spec * 255).astype(np.uint8))
    img = img.resize(IMG_SIZE, Image.BILINEAR)
    return np.array(img)

# ---------- Process Single Audio ----------
def audio_to_rgb(audio_path, save_path):
    y, sr = librosa.load(audio_path, sr=SR)

    # Spectrogram
    stft = librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH)
    spec_db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)
    R = resize(normalize(spec_db))

    # Mel-Spectrogram
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=N_FFT,
        hop_length=HOP_LENGTH, n_mels=N_MELS
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    G = resize(normalize(mel_db))

    # GFCC (Gammatone)
    gtg = gtgram(
        y, sr,
        window_time=0.025,
        hop_time=0.010,
        channels=GT_CHANNELS,
        f_min=50
    )
    gfcc_log = np.log(gtg + 1e-10)
    B = resize(normalize(gfcc_log))

    # Stack RGB
    rgb = np.stack([R, G, B], axis=-1)
    Image.fromarray(rgb, "RGB").save(save_path)

# ---------- Batch Conversion ----------
for root, _, files in os.walk(INPUT_ROOT):
    for file in tqdm(files):
        if file.lower().endswith(".wav"):
            audio_path = os.path.join(root, file)

            # Preserve class folder
            class_name = os.path.basename(root)
            save_dir = os.path.join(OUTPUT_ROOT, class_name)
            os.makedirs(save_dir, exist_ok=True)

            save_path = os.path.join(
                save_dir, file.replace(".wav", ".png")
            )

            try:
                audio_to_rgb(audio_path, save_path)
            except Exception as e:
                print(f"❌ Error processing {audio_path}: {e}")

print("✅ All audio files converted to RGB images (224×224)")


0it [00:00, ?it/s]
100%|████████████████████████████████████████████████████████████████████████████████| 192/192 [00:45<00:00,  4.25it/s]

✅ All audio files converted to RGB images (224×224)
